# 02. Model Training & Evaluation

## Overview
This notebook serves as the main entry point for:
1.  **Verifying the Hardware** (GPU availability).
2.  **Training the ResNet50 Model** (executing the training pipeline).
3.  **Evaluating Performance** using professional diagnostic plots (Confusion Matrix, ROC, Reliability Curves).

## Objectives
-   Confirm CUDA acceleration is active.
-   Train the model on the processed dataset.
-   Assess generalization performance on the validation set.
-   Visualize class-wise metrics and prediction confidence.

In [ ]:
# 1. Imports & Setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import logging
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import PROJECT_ROOT, MODELS_DIR, REPORTS_DIR
from src.visualization.plots import (
    set_plot_style,
    plot_learning_curves,
    plot_model_diagnostics,
    plot_confidence_abstention_panel
)

# Configure Styling
set_plot_style()
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

print(f"Project Root: {PROJECT_ROOT}")

In [ ]:
# 2. Hardware Verification
print(f"PyTorch Version: {torch.__version__}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    device_count = torch.cuda.device_count()
    print(f"\u2705 SUCCESS: GPU is available!")
    print(f"   Device: {device_name}")
    print(f"   Count: {device_count}")
    print(f"   Cuda Version: {torch.version.cuda}")
else:
    print("\u26A0 WARNING: GPU not detected. Training will be extremely slow on CPU.")
    print("   Please check your environment configuration.")

In [ ]:
# 3. Training Loop Execution
# Automatic logic: If artifacts exist, skip training. Else, run training pipeline.

FORCE_RETRAIN = False # Set to True to re-run training even if results exist

HISTORY_PATH = REPORTS_DIR / "training_history.joblib"
PREDICTIONS_PATH = REPORTS_DIR / "predictions.csv"

if (HISTORY_PATH.exists() and PREDICTIONS_PATH.exists()) and not FORCE_RETRAIN:
    print(f"Found existing training artifacts at {REPORTS_DIR}")
    print("   Skipping training loop. Loading results...")
else:
    if FORCE_RETRAIN:
        print("FORCE_RETRAIN is True. Overwriting existing results...")
    else:
        print("Metrics not found. Starting training pipeline...")
        
    print("--> Executing: python -m src.models.train_model")
    # Execute Training Script
    !python -m src.models.train_model
    
    # Reload to verify
    if HISTORY_PATH.exists():
        print("Training finished successfully.")
    else:
        raise RuntimeError("Training script finished but artifacts were not found! Check terminal for errors.")

In [ ]:
# 4. Evaluation: Load Artifacts

# Paths to artifacts (adjust timestamp/folder if necessary)
# By default, the script saves to models/ and reports/
# We look for the most recent run artifacts if exact path not specified

HISTORY_PATH = REPORTS_DIR / "training_history.joblib"
PREDICTIONS_PATH = REPORTS_DIR / "predictions.csv"

if HISTORY_PATH.exists():
    print(f"Loading history from: {HISTORY_PATH}")
    history = joblib.load(HISTORY_PATH)
    
    # Plot Learning Curves
    plot_learning_curves(history, save_folder="evaluation")
    plt.show()
else:
    print(f"History file not found at {HISTORY_PATH}. Run training first.")

In [ ]:
# 5. Diagnostic Panel (Confusion Matrix & ROC)

if PREDICTIONS_PATH.exists():
    print(f"Loading predictions from: {PREDICTIONS_PATH}")
    results_df = pd.read_csv(PREDICTIONS_PATH)
    
    # Ensure columns match expected names
    # train_model.py should output 'image_name', 'true', 'pred', 'prob_0', 'prob_1'...
    print(f"Columns: {results_df.columns.tolist()}")
    
    # Generate Diagnostics
    plot_model_diagnostics(results_df, n_classes=5, save_folder="evaluation")
    plt.show()
else:
    print(f"Predictions file not found at {PREDICTIONS_PATH}")

In [ ]:
# 6. Reliability & Abstention Analysis

if PREDICTIONS_PATH.exists():
    # Analyze Confidence
    # A threshold of 0.7 means we only classify if max probability > 70%
    plot_confidence_abstention_panel(results_df, confidence_threshold=0.7, save_folder="evaluation")
    plt.show()

In [ ]:
# 7. Sample Predictions Audit
from src.visualization.plots import plot_image_grid
import cv2

# Function to load images based on paths in dataframe (if available) or raw loading
# For demonstration, we just need to load a few images from the test set.

# Note: results_df from train_model.py typically has 'image_name'.
# We need to construct the full path.

if PREDICTIONS_PATH.exists() and 'image_name' in results_df.columns:
    # Construct path (Adjust based on your data structure, e.g. data/train/)
    from src.config import PROCESSED_DATA_DIR, DATA_DIR
    
    # Identify False Predictions
    errors = results_df[results_df['true'] != results_df['pred']]
    
    if len(errors) > 0:
        sample_errors = errors.sample(n=min(8, len(errors)), random_state=42)
        
        images = []
        labels = []
        preds = []
        titles = []
        
        for idx, row in sample_errors.iterrows():
            img_path = DATA_DIR / "train" / f"{row['image_name']}.jpeg"
            if not img_path.exists():
                # Try adding extension or different path
                pass
            
            # Simple load (using opencv or PIL)
            # We use a placeholder logic here assuming path exists
            try:
                img = cv2.imread(str(img_path))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append(img)
                labels.append(row['true'])
                preds.append(row['pred'])
                titles.append(f"True: {row['true']} | Pred: {row['pred']}")
            except:
                pass
        
        if images:
            print("Visualizing Error Cases:")
            plot_image_grid(images, labels, preds=preds, titles=titles, title="Failure Analysis (False Predictions)")
            plt.show()
    else:
        print("Model has 100% accuracy! No errors to visualize.")